# Fundamentals of Machine Learning — Exercise 11
## Regression Models, Metrics, Residuals, and Transfer Across Datasets

Regression is not “classification with continuous labels”. The target scale, error costs, residual structure and model assumptions all change how a result should be evaluated.

This exercise revisits regression as a complete modelling task and compares the same modelling ideas across **three real-world datasets**:

1. Petrol Consumption,
2. Diabetes,
3. House Prices.

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- distinguish MAE, RMSE, MAPE and $R^2$,
- explain why the choice of metric depends on the application,
- interpret residual plots rather than using one score only,
- compare one-feature and multivariable linear regression,
- explain L1/Lasso and L2/Ridge regularisation,
- explain why scaling matters when coefficients are penalised,
- use regression versions of tree and ensemble models,
- compare models using cross-validation rather than one lucky split,
- recognise that the best model can change across datasets and metrics,
- use a leakage-safe pipeline,
- formulate a cautious model recommendation with uncertainty and limitations,
- use AI to challenge a metric/model choice after making your own decision.

**Recommended duration:** 110–120 minutes.  
The three-dataset comparison may be completed as homework.

## How to work in this exercise

Most model code is prepared. Your task is to:

1. predict which error measure or model is appropriate,
2. inspect residuals and not only a leaderboard number,
3. compare models under the same validation protocol,
4. explain regularisation in terms of coefficients and generalisation,
5. transfer the reasoning to datasets with different sample sizes and target scales.

A lower RMSE is useful evidence. It is not a complete scientific explanation.

## 0. Imports

In [ ]:
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import TransformedTargetRegressor
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## 1. Metrics answer different questions

Consider two models with the same number of predictions. Before computing, decide which error vector should be punished more strongly by **RMSE** and why.

In [ ]:
y_true_demo = np.array([100, 100, 100, 100])
pred_a = np.array([90, 90, 110, 110])       # four moderate errors
pred_b = np.array([100, 100, 100, 120])      # one large error

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero]-y_pred[nonzero])/y_true[nonzero]))*100
    return {'MAE':mae, 'RMSE':rmse, 'MAPE (%)':mape, 'R2':r2}

pd.DataFrame([
    {'model':'A', **regression_metrics(y_true_demo,pred_a)},
    {'model':'B', **regression_metrics(y_true_demo,pred_b)},
]).round(3)

### Activity 1 — Match metric to meaning

Complete:

- MAE is easiest to explain as __________.
- RMSE reacts more strongly to __________ because __________.
- MAPE is attractive because __________, but problematic when targets are __________.
- $R^2<0$ means __________.

Give one application where MAE would be more useful than RMSE, and one where a large individual error should be penalised strongly.

## 2. Dataset 1 — Petrol Consumption

This is the same small real-world dataset used in the original regression exercise: 48 US states, four predictors and petrol consumption as the target.

The data are embedded so the notebook runs without downloading a file.

In [ ]:
PETROL_CSV = r"""Petrol_tax,Average_income,Paved_Highways,Population_Driver_licence(%),Petrol_Consumption
9,3571,1976,0.525,541
9,4092,1250,0.572,524
9,3865,1586,0.58,561
7.5,4870,2351,0.529,414
8,4399,431,0.544,410
10,5342,1333,0.571,457
8,5319,11868,0.451,344
8,5126,2138,0.553,467
8,4447,8577,0.529,464
7,4512,8507,0.552,498
8,4391,5939,0.53,580
7.5,5126,14186,0.525,471
7,4817,6930,0.574,525
7,4207,6580,0.545,508
7,4332,8159,0.608,566
7,4318,10340,0.586,635
7,4206,8508,0.572,603
7,3718,4725,0.54,714
7,4716,5915,0.724,865
8.5,4341,6010,0.677,640
7,4593,7834,0.663,649
8,4983,602,0.602,540
9,4897,2449,0.511,464
9,4258,4686,0.517,547
8.5,4574,2619,0.551,460
9,3721,4746,0.544,566
8,3448,5399,0.548,577
7.5,3846,9061,0.579,631
8,4188,5975,0.563,574
9,3601,4650,0.493,534
7,3640,6905,0.518,571
7,3333,6594,0.513,554
8,3063,6524,0.578,577
7.5,3357,4121,0.547,628
8,3528,3495,0.487,487
6.58,3802,7834,0.629,644
5,4045,17782,0.566,640
7,3897,6385,0.586,704
8.5,3635,3274,0.663,648
7,4345,3905,0.672,968
7,4449,4639,0.626,587
7,3656,3985,0.563,699
7,4300,3635,0.603,632
7,3745,2611,0.508,591
6,5215,2302,0.672,782
9,4476,3942,0.571,510
7,4296,4083,0.623,610
7,5002,9794,0.593,524"""
petrol = pd.read_csv(StringIO(PETROL_CSV))
petrol.head()

In [ ]:
petrol.describe().T

### Predict from domain and EDA

Before fitting any model:

1. Which feature do you expect to have the strongest linear relationship with `Petrol_Consumption`?
2. Which feature scale is numerically largest?
3. Why does numerical scale not automatically equal predictive importance?

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
sns.heatmap(petrol.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0, ax=axes[0])
sns.scatterplot(data=petrol, x='Population_Driver_licence(%)', y='Petrol_Consumption', ax=axes[1])
axes[1].set_title('A strong candidate relationship')
plt.tight_layout(); plt.show()

## 3. One-feature versus multivariable linear regression

We use one fixed train/test split here only to visualise residuals. Later model selection will use cross-validation.

In [ ]:
X_petrol = petrol.drop(columns='Petrol_Consumption')
y_petrol = petrol['Petrol_Consumption']
Xtr, Xte, ytr, yte = train_test_split(X_petrol, y_petrol, test_size=0.25, random_state=RANDOM_STATE)

single_feature = 'Population_Driver_licence(%)'
lin_single = LinearRegression().fit(Xtr[[single_feature]], ytr)
lin_multi = LinearRegression().fit(Xtr, ytr)

p_single = lin_single.predict(Xte[[single_feature]])
p_multi = lin_multi.predict(Xte)

pd.DataFrame([
    {'model':'one feature', **regression_metrics(yte,p_single)},
    {'model':'all four features', **regression_metrics(yte,p_multi)},
]).round(3)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4.8))
for ax, name, pred in [
    (axes[0], 'One-feature linear model', p_single),
    (axes[1], 'Multivariable linear model', p_multi),
]:
    residual = yte - pred
    ax.scatter(pred, residual)
    ax.axhline(0, linestyle='--')
    ax.set_xlabel('Predicted petrol consumption')
    ax.set_ylabel('Residual = observed - predicted')
    ax.set_title(name)
plt.tight_layout(); plt.show()

### Activity 2 — Read residuals

1. Which model has lower errors on this split?
2. Do residuals look randomly scattered around zero?
3. Identify one observation/model combination with a large residual.
4. Why is a residual plot evidence that a single aggregate metric cannot provide?

## 4. Regularisation: Ridge and Lasso

Linear regression estimates coefficients to minimise squared error. Ridge and Lasso add a penalty:

- **Ridge (L2)** shrinks coefficients smoothly,
- **Lasso (L1)** can shrink some coefficients exactly to zero.

Because the penalty acts on coefficient magnitude, feature scaling matters.

In [ ]:
regularised_models = {
    'Linear regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LinearRegression()),
    ]),
    'Ridge alpha=10': Pipeline([
        ('scale', StandardScaler()),
        ('model', Ridge(alpha=10)),
    ]),
    'Lasso alpha=10': Pipeline([
        ('scale', StandardScaler()),
        ('model', Lasso(alpha=10, max_iter=10000)),
    ]),
}

coef_rows=[]
for name, pipe in regularised_models.items():
    pipe.fit(Xtr,ytr)
    coef = pipe.named_steps['model'].coef_
    for feature,value in zip(Xtr.columns,coef):
        coef_rows.append({'model':name,'feature':feature,'coefficient':value})
coef_df = pd.DataFrame(coef_rows)

plt.figure(figsize=(11,5))
sns.barplot(data=coef_df, x='feature', y='coefficient', hue='model')
plt.xticks(rotation=25)
plt.title('Coefficients after standardising predictors')
plt.tight_layout(); plt.show()

### Activity 3 — Explain regularisation from the coefficients

1. Which method shrinks coefficients most strongly?
2. Does Lasso set any coefficient close to zero here?
3. Why would comparing raw coefficient magnitudes before scaling be misleading?
4. Why does a smaller coefficient not mean that the feature is scientifically unimportant?

## 5. Regression versions of familiar model families

Many model ideas previously used for classification also have regression variants. We compare:

- linear regression,
- Ridge,
- Random Forest regression,
- histogram gradient boosting regression.

We use the same 5-fold cross-validation splits for each model.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

petrol_models = {
    'Linear': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
    'Ridge': Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=10))]),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=7, min_samples_leaf=4, random_state=RANDOM_STATE),
}

scoring = {'MAE':'neg_mean_absolute_error', 'RMSE':'neg_root_mean_squared_error', 'R2':'r2'}
rows=[]
for name,model in petrol_models.items():
    scores = cross_validate(model, X_petrol, y_petrol, cv=cv, scoring=scoring)
    rows.append({
        'model':name,
        'CV MAE':-scores['test_MAE'].mean(),
        'CV RMSE':-scores['test_RMSE'].mean(),
        'CV R2':scores['test_R2'].mean(),
        'R2 std':scores['test_R2'].std(),
    })
petrol_cv = pd.DataFrame(rows).sort_values('CV RMSE')
petrol_cv.round(3)

### Activity 4 — Model choice on a small dataset

1. Which model has the best mean RMSE?
2. Is the ranking identical under MAE and $R^2$?
3. How large is the fold-to-fold variation?
4. Why should the tiny sample size make you cautious about declaring a universal winner?

## 6. Dataset 2 — Diabetes

The built-in diabetes dataset has 442 observations and 10 baseline measurements. The target is a quantitative disease-progression measure one year later.

We reuse the same modelling protocol.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X_diabetes = diabetes.data
y_diabetes = diabetes.target
print(X_diabetes.shape)
X_diabetes.head()

## 7. Dataset 3 — House Prices

House Prices has already appeared earlier in the course. We now reuse its numeric variables for a genuine regression task.

The cell first tries the course dataset URL. If internet access is unavailable, this section can be run later in Colab.

In [ ]:
HOUSE_URL = (
    'https://raw.githubusercontent.com/rasvob/'
    'VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/'
    'master/datasets/zsu_cv1_data.csv'
)

house = pd.read_csv(HOUSE_URL)
selected_house_features = [
    'OverallQual','OverallCond','YearBuilt','GrLivArea','LotArea','BedroomAbvGr'
]
X_house = house[selected_house_features].copy()
y_house = house['SalePrice'].copy()
print(X_house.shape)
X_house.head()

## 8. Transfer the same models across three datasets

We define a reusable evaluator. Each dataset keeps its own target scale, so MAE/RMSE values are **not comparable across datasets**. The question is how model rankings and uncertainty change within each dataset.

In [ ]:
def compare_regressors(X, y, dataset_name):
    models = {
        'Linear': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', LinearRegression()),
        ]),
        'Ridge': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', Ridge(alpha=10)),
        ]),
        'Random Forest': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(
                n_estimators=250, min_samples_leaf=2, random_state=RANDOM_STATE
            )),
        ]),
        'Hist Gradient Boosting': Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('model', HistGradientBoostingRegressor(
                max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
                min_samples_leaf=5, random_state=RANDOM_STATE
            )),
        ]),
    }
    cv_local = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    result=[]
    for model_name,model in models.items():
        scores = cross_validate(model, X, y, cv=cv_local, scoring=scoring)
        result.append({
            'dataset':dataset_name,
            'model':model_name,
            'MAE':-scores['test_MAE'].mean(),
            'RMSE':-scores['test_RMSE'].mean(),
            'R2':scores['test_R2'].mean(),
            'R2 std':scores['test_R2'].std(),
        })
    return pd.DataFrame(result)

all_results = pd.concat([
    compare_regressors(X_petrol, y_petrol, 'Petrol'),
    compare_regressors(X_diabetes, y_diabetes, 'Diabetes'),
    compare_regressors(X_house, y_house, 'House Prices'),
], ignore_index=True)

all_results.round(3)

In [ ]:
rank_table = all_results.copy()
rank_table['RMSE rank within dataset'] = rank_table.groupby('dataset')['RMSE'].rank(method='min')
rank_table.sort_values(['dataset','RMSE rank within dataset']).round(3)

### Activity 5 — What transfers and what does not?

For each dataset identify the best RMSE model, then answer:

1. Is one model best on all three datasets?
2. Where is a linear/regularised model competitive?
3. Where do nonlinear ensembles help?
4. Which dataset has the most unstable $R^2$ across folds?
5. Why must you not compare a Petrol RMSE numerically with a House Price RMSE?

## 9. Target transformation: a useful but consequential choice

House prices are often right-skewed. A log target changes the optimisation emphasis: relative differences become more important than absolute currency differences.

We compare Random Forest on the original target with a log-transformed target wrapper.

In [ ]:
rf_plain = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE)),
])

rf_log_target = TransformedTargetRegressor(
    regressor=rf_plain,
    func=np.log1p,
    inverse_func=np.expm1,
)

for name,model in [('RF original target',rf_plain), ('RF log1p target',rf_log_target)]:
    scores = cross_validate(model, X_house, y_house, cv=cv, scoring=scoring)
    print(name, {
        'MAE': round(-scores['test_MAE'].mean(),1),
        'RMSE': round(-scores['test_RMSE'].mean(),1),
        'R2': round(scores['test_R2'].mean(),3),
    })

### Activity 6 — Interpret the target transformation

1. Did the transformation improve every metric?
2. Why can the answer differ between MAE and RMSE?
3. What error behaviour is emphasised after a log transform?
4. Why should a target transform be chosen for modelling reasons, not only because it improves one number?

## 10. Guided AI study activity — challenge your metric/model choice

Write your own recommendation for **one** dataset first. Then use:

```text
Act as a regression-method reviewer. Do not choose a model for me and do not rewrite my answer.
Ask me exactly three short questions, one at a time:
1. What real-world error does my chosen metric represent?
2. What residual or cross-validation evidence supports my model choice?
3. Which assumption, uncertainty or alternative model could weaken my conclusion?
Wait for my answer after each question.
```

Close the chat and record only:

**One useful AI question:**  

**One claim I revised:**  

**Evidence used for the revision:**

## 11. Assessed task — Three-dataset regression memo (2 points)

Choose **one** of the three datasets for your main recommendation, but use the other two as comparison context.

Submit:

1. target and practical meaning of the prediction task,
2. primary metric and why it fits the task,
3. at least three models compared under the same cross-validation protocol,
4. mean performance and variability,
5. one residual or coefficient/feature observation,
6. recommended model,
7. one reason a different model wins or loses on another dataset,
8. one limitation,
9. optional verified AI note.

### Marking

| Criterion | Points |
|---|---:|
| Metric and validation protocol are appropriate | 0.50 |
| Model comparison uses evidence, not only ranking | 0.50 |
| Recommendation and cross-dataset transfer are reasoned | 0.50 |
| Limitation / uncertainty / verified AI note is meaningful | 0.50 |

### Your regression memo

**Dataset and target:**  

**Primary metric and why:**  

**Models compared:**  

**Cross-validation evidence:**  

**Residual / coefficient / feature observation:**  

**Recommendation:**  

**Cross-dataset comparison:**  

**Limitation:**  

**Optional verified AI note:**

# End-of-lab self-check

- [ ] I can distinguish MAE, RMSE, MAPE and R².
- [ ] I can explain why one metric may matter more in a particular application.
- [ ] I inspect residuals rather than trusting one score.
- [ ] I can explain L1 and L2 regularisation.
- [ ] I know why penalised linear models should usually be fitted after scaling.
- [ ] I can name regression counterparts of tree/ensemble classifiers.
- [ ] I compare models under the same cross-validation protocol.
- [ ] I understand that model rankings can change across datasets.
- [ ] I do not compare RMSE values across targets with different units/scales.
- [ ] I can explain why a target transformation changes the modelling objective.
- [ ] I can defend a regression recommendation with evidence and a limitation.